In [35]:
#!/usr/bin/env python

import rospy
import actionlib
import actionlib.msg
import sys
import threading 

from geometry_msgs.msg import PoseStamped
from nav_msgs.msg import Odometry
from assignment_2_2024.msg import PlanningAction, PlanningGoal
from assignment2_ros1.msg import RobotInfo
from sensor_msgs.msg import LaserScan

import ipywidgets as widgets
from IPython.display import display
import os
import platform

%matplotlib widget

import matplotlib.pyplot as plt 
import tf

from tf.transformations import quaternion_matrix 
import numpy as np
from matplotlib.animation import FuncAnimation

import time
from ipywidgets import Button, Layout, ButtonStyle, GridBox, VBox, HBox

In [36]:
# Global variable to store the latest feedback
feedbackData = None 
distanceFromObstacle = 0.0

robotStatus = RobotInfo()
distanceWid = widgets.Label()
odomWid = widgets.Label()

In [37]:

def feedbackHandler(feedback):
    """Callback to handle feedback from the action server."""
    global feedbackData
    feedbackData = feedback

def sendGoal(client, xTarget, yTarget):
    """Sends a goal to the action server."""
    goal = PlanningGoal()
    goal.target_pose = PoseStamped()
    goal.target_pose.pose.position.x = xTarget
    goal.target_pose.pose.position.y = yTarget

    client.send_goal(goal, feedback_cb=feedbackHandler)
    rospy.loginfo("Goal has been sent to the action server.")

def publishStatus(odomMsg):
    """Publishes the robot's current position and velocity."""
    global robotStatus 
    global xdata
    global ydata
    
    robotStatus.x = odomMsg.pose.pose.position.x
    robotStatus.y = odomMsg.pose.pose.position.y
    robotStatus.velX = odomMsg.twist.twist.linear.x
    robotStatus.velZ = odomMsg.twist.twist.angular.z
    
    ydata.append(odomMsg.pose.pose.position.y)
    xdata.append(odomMsg.pose.pose.position.x)
    
    statusPublisher.publish(robotStatus)

In [38]:
def sendTarget(b,xTarget,yTarget, client):
    print(f"Targets:{xTarget},{yTarget}")
    
    # Initialize the action client and wait for the server
    
    client.wait_for_server()
    sendGoal(client, xTarget, yTarget)
    
def quitBtnClick(b,client):
    if client.get_state() not in [actionlib.GoalStatus.SUCCEEDED, actionlib.GoalStatus.ABORTED, actionlib.GoalStatus.PREEMPTED]:
        rospy.loginfo("Cancelling the current goal.")
        client.cancel_goal()
    else:
        rospy.loginfo("The goal has already been achieved.")
def feedbackBtnClick(b):
    rospy.loginfo("Requesting feedback...")
    if feedbackData is None:
        rospy.loginfo("No feedback received yet.")
    else:
        rospy.loginfo("Latest feedback: %s", feedbackData)
        
def exitBtnClick(b,client):
    if client.get_state() not in [actionlib.GoalStatus.SUCCEEDED, actionlib.GoalStatus.ABORTED, actionlib.GoalStatus.PREEMPTED]:
        rospy.loginfo("Cancelling goal and shutting down.")
        client.cancel_goal()
    else:
        rospy.loginfo("Shutting down.")

In [39]:
def obstDist(laserMsg):
    global distanceFromObstacle
    distanceFromObstacle = min(min(laserMsg.ranges), 10)   

In [40]:
def updateValues():
    display(distanceWid)
    display(odomWid)
    while not rospy.is_shutdown():
        distanceWid.value = f"Distance: {distanceFromObstacle:.2f} m"
        pos = f"Position: (x: {robotStatus.x:.2f}, y: {robotStatus.y:.2f})"
        vel = f"Velocity: lin {robotStatus.velX:.2f}, ang {robotStatus.velZ:.2f}"
        odomWid.value = f"{pos} | {vel}"
        
        time.sleep(0.5)
        
        

In [41]:
def plot_init():
    ax.set_xlim(-10, 10)
    ax.set_ylim(-10, 10)
    return ln,

def plot_update(frame):
    ln.set_data(xdata, ydata)
    return ln,

In [42]:
if __name__ == '__main__':
    try:
        rospy.init_node("robotActionClient")
        rospy.sleep(2)

        statusPublisher = rospy.Publisher("/robot_status", RobotInfo, queue_size=10)
        rospy.Subscriber("/odom", Odometry, publishStatus)
        rospy.Subscriber("/scan",LaserScan, obstDist)
        rate = rospy.Rate(10)
        
        #Global variables for the plot

        fig, ax = plt.subplots()
        xdata, ydata = [], []
        ln, = plt.plot([], [], 'ro')
        #Setup the thread
        
        t = threading.Thread(target=updateValues)
        t.daemon = True
        t.start()
        
        
        rate.sleep()
        # Request target coordinates from the user
        
        actionClient = actionlib.SimpleActionClient('reaching_goal', PlanningAction)

        #Get coordinates
        
        print("Set the xTarget and the yTarget and then press the start button to set the target")
        xCoordWid = widgets.FloatText(value=6,description='xTarget:',disabled=False)
        yCoordWid = widgets.FloatText(value=6,description='yTarget:',disabled=False)
        display(xCoordWid)
        display(yCoordWid)
        
        startBtn = widgets.Button(description = "Start")
        startBtn.style.button_color = 'lightgreen'
        display(startBtn)
        
        xTarget = xCoordWid.value
        yTarget = yCoordWid.value
        
        # Use partial to bind local variables to the test function
        def startBtnClick(b):
            xTarget = xCoordWid.value
            yTarget = yCoordWid.value
            
            #TODO: check if the coordinates are in the boundaries
            
            sendTarget(b, xTarget, yTarget, actionClient)

        startBtn.on_click(startBtnClick)
        
        quitBtn = Button(description='Quit',
        layout=Layout(width='auto', align="center",
        grid_area='quitBtn'),
        style=ButtonStyle(button_color='salmon'))
        feedbackBtn = Button(description='Feedback',
        layout=Layout(width='auto', grid_area='feedbackBtn'),
        style=ButtonStyle(button_color='moccasin'))
        exitBtn = Button(description='Exit',
        layout=Layout(width='auto', grid_area='exitBtn'),
        style=ButtonStyle(button_color='lightblue'))
                  
        horizontalBox = HBox([quitBtn,feedbackBtn,exitBtn])
        
        display(horizontalBox)
        
        def stopBtnClick(b):
            quitBtnClick(b, actionClient)
            
        def exitButton(b):
            exitBtnClick(b, actionClient)

        quitBtn.on_click(stopBtnClick)
        feedbackBtn.on_click(feedbackBtnClick)
        exitBtn.on_click(exitButton)
        
        # matplotlib
       
        ani = FuncAnimation(fig, plot_update, init_func=plot_init)
        plt.show()
        
    except rospy.ROSInterruptException:
        rospy.logerr("Action client was interrupted.")
        sys.exit(1)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Label(value='Distance: 1.04 m')

Label(value='Position: (x: -0.12, y: 1.01) | Velocity: lin -0.00, ang -0.00')

Set the xTarget and the yTarget and then press the start button to set the target


FloatText(value=6.0, description='xTarget:')

FloatText(value=6.0, description='yTarget:')

Button(description='Start', style=ButtonStyle(button_color='lightgreen'))

Targets:6.0,6.0
[INFO] [1746448279.894387]: Goal has been sent to the action server.
